<h1>Downloading PACE OCI Products</h1>

<p>Equivalent to the Sentinel-3 OLCI download and analysis pipeline, adapted for NASA PACE OCI.<br>
<br>Main differences:<br>
<br>--- Authentication with earthaccess.login() instead of manual token refresh<br>
--- With Sen-3, chl-a and PAR were both inside the same downloaded file. For PACE, download the BGC file for chlor_a, and a separate a PAR file for par<br>
--- Single .nc file per granule (not .SEN3 folder structure)<br>
--- For PACE, there are 'subfolders' inside the NetCDF files. Sen-3 OLCI was flat; all variables sat at one level. PACE .nc files have subfolders: lat/lon are in navigation_data and science vars are in geophysical_data<br>
<br>--- chlor_a replaces CHL_OC4ME and CHL_NN <br>
--- For PAR, using PACE's ipar_planar_above: light measured just above the ocean surface. Most similar to the PAR derived from Sen-3<br>
--- PAR units: multiplied ipar_planar_above by 1×10⁶ to obtain same units before comparing directly to OLCI PAR values<br>
<br>--- Quality flags: l2_flags bitmask inside the .nc file too, under geophysical_data<br>
</p>

In [21]:
import earthaccess
import os
import netCDF4 as nc
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pyresample import geometry, kd_tree
from pyproj import CRS, Transformer
from datetime import datetime
from collections import defaultdict
import json
import re

In [22]:
# CONFIGURATION #

# McMurdo Sound bounding box (lon_min, lat_min, lon_max, lat_max)
# earthaccess bbox format is (W, S, E, N)
aoi_bbox = (162.0, -78.5, 168.0, -77.0)
aoi_lat_min = -78.5
aoi_lat_max = -77.0
aoi_lon_min = 162.0
aoi_lon_max = 168.0

# PACE OCI Level 2 Products
# BGC suite: contains chlor_a
# PAR suite: contains par (daily PAR) and ipar (instantaneous PAR)
short_name_BGC = "PACE_OCI_L2_BGC"   # for chlor_a
short_name_PAR = "PACE_OCI_L2_PAR"   # for ipar

# science variables to extract per suite
science_vars = {
    short_name_BGC: {
        "chlor_a": "CHLOR_A", # chlorophyll-a (mg m^-3)
    },
    short_name_PAR: {
        "ipar_planar_above": "PAR", # instantaneous, above-surface - comparable to Sen-3 OLCI PAR
    },
}

# PACE l2_flags has same bitmask system used by Sen-3 level 2 products
# source: https://oceancolor.gsfc.nasa.gov/resources/atbd/ocl2flags/

# all applicable quality flags:
# not LAND (pixel is over land)
# not CLDICE  (cloud or ice)
# not HIGLINT (sunglint: reflectance exceeds threshold)
# not ATMFAIL (atmospheric correction failure)
# not HILT    (observed radiance very high or saturated)
# not STRAYLIGHT (probable stray light contamination)
# not NAVFAIL (navigation failure)
# not HISOLZEN (solar zenith exceeds threshold — relevant for polar regions)

# variable-specific flags:
# CHLFAIL (chlorophyll algorithm failure)
# PRODFAIL (failure in any product)

# can't find recommended flags for PACE L2 products
# selected based on similarities withSen-3 flags

common_invalid_flags = [
    "LAND",         # no specific WATER flag, so we choose not LAND pixels
    "CLDICE",       # similar to Sen-3 CLOUD and SNOW_ICE
    "HIGLINT",      # similar to Sen-3 HIGHGLINT
    "ATMFAIL",      # similar to Sen-3 AC_FAIL
    "HILT",         # similar to Sen-3 SATURATED
    "STRAYLIGHT",   # similar to Sen-3 COSMETIC / SUSPECT
    "NAVFAIL",      # similar to Sen-3 INVALID
    "HISOLZEN",     # high solar zenith — important at polar latitudes
]

variable_flags = {
    "CHLOR_A": ["CHLFAIL", "PRODFAIL"],   # similar to Sen-3 OC4ME_FAIL
    "PAR":     ["PRODFAIL"],
}

# target CRS and resolution for optional GeoTIFF output
target_crs = "EPSG:5479" # RSRGD2000 / MSLC2000
target_res_metres = 1200 # PACE L2 products native resolution (https://www.earthdata.nasa.gov/dashboard/data-catalog/OCI_PACE_Chlorophyll_a)

# AUTHENTICATION #

# log in to NASA Earthdata using earthaccess
# call once at the start of the session, with no token refresh needed
def authenticate():
    earthaccess.login(strategy="interactive", persist=True)
    print("Earthdata login successful.")

# DOWNLOADING PACE OCI L2 PRODUCTS #

# search for PACE OCI L2 granules that cover the AOI and date range
# search twice, once for the BGC suite and once for the PAR suite (since PACE has products for each of them)
# pair these suites by sensing time
def search_pace_oci(start_date, end_date, bbox=aoi_bbox):

    time_span = (start_date, end_date)
    
    print(f"\nSearching {short_name_BGC} from {start_date} to {end_date}.")
    bgc_results = earthaccess.search_data(
        short_name=short_name_BGC,
        temporal=time_span,
        bounding_box=bbox,
    )
    print(f"Found {len(bgc_results)} BGC granule(s).")

    print(f"\nSearching {short_name_PAR} from {start_date} to {end_date}.")
    par_results = earthaccess.search_data(
        short_name=short_name_PAR,
        temporal=time_span,
        bounding_box=bbox,
    )
    print(f"Found {len(par_results)} PAR granule(s).")

    return {"bgc": bgc_results, "par": par_results}

# download all granules from search results into downloads_folder
def download_granules(search_results, downloads_folder):

    os.makedirs(downloads_folder, exist_ok=True)
    downloaded = {"bgc": [], "par": []}

    for suite, results in search_results.items():
        if not results:
            print(f"No {suite.upper()} granules to download.")
            continue

        print(f"\nDownloading {len(results)} {suite.upper()} granule(s).")
        
        # earthaccess.download() handles auth, retries, and skips already-downloaded files
        files = earthaccess.download(results, downloads_folder)
        downloaded[suite].extend(files)
        print(f"{suite.upper()}: {len(files)} file(s) saved to {downloads_folder}")

    return downloaded

In [17]:
# MAIN CODE #
 
authenticate()
 
search_results = search_pace_oci(
    start_date="2025-12-01",
    end_date="2025-12-05",   # end date is inclusive
)
 
total = len(search_results["bgc"]) + len(search_results["par"])
print(f"\nFound {total} total granule(s) ({len(search_results['bgc'])} BGC, "
      f"{len(search_results['par'])} PAR).")
 
downloads_folder = "/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_0_RAWDATA/PACE"
 
confirm = input("\nProceed with download? (y/n): ")
if confirm.lower() == "y":
    download_granules(search_results, downloads_folder)
    print("\nAll downloads complete.")
else:
    print("Download cancelled.")

Earthdata login successful.

Searching PACE_OCI_L2_BGC from 2025-12-01 to 2025-12-05.
Found 60 BGC granule(s).

Searching PACE_OCI_L2_PAR from 2025-12-01 to 2025-12-05.
Found 60 PAR granule(s).

Found 120 total granule(s) (60 BGC, 60 PAR).



QUEUEING TASKS | : 100%|██████████| 60/60 [00:00<00:00, 4470.90it/s]
PROCESSING TASKS | : 100%|██████████| 60/60 [03:05<00:00,  3.10s/it]
COLLECTING RESULTS | : 100%|██████████| 60/60 [00:00<00:00, 343326.38it/s]


BGC: 60 file(s) saved to /Users/gwyneth/Desktop/Masters_Thesis/LEVEL_0_RAWDATA/PACE



QUEUEING TASKS | : 100%|██████████| 60/60 [00:00<00:00, 3413.15it/s]
PROCESSING TASKS | : 100%|██████████| 60/60 [06:50<00:00,  6.84s/it]
COLLECTING RESULTS | : 100%|██████████| 60/60 [00:00<00:00, 284359.59it/s]

PAR: 60 file(s) saved to /Users/gwyneth/Desktop/Masters_Thesis/LEVEL_0_RAWDATA/PACE

All downloads complete.


<h1>Processing Downloaded OCI Products</h1>

<p>Step 1: Read NetCDF files and write NetCDF with embedded lat/lon<br>
Step 2: Apply l2_flags and write quality-masked NetCDF files<br>
Step 3: Extract pixels within AOI bounding box<br>
Step 4: Calculate daily and weekly statistics<br>
Step 5 (optional): Resample to GeoTIFF and reproject to EPSG:5479 (MSLC2000) for visualisation</p>

In [23]:
# Step 1: Read downloaded NetCDF files and embedded them with lat/lon coordinates

# extract a granule's name
# PACE files naming format: PACE_OCI.<date+timestamp>.L2.<suite>.<version>.nc
def get_pace_granule_name(nc_path):
    return os.path.splitext(os.path.basename(nc_path))[0]

# parse the sensing date from the .nc filename
def parse_sensing_date_pace(nc_path):

    basename = os.path.basename(nc_path)
    match = re.search(r"\.(\d{8})T\d{6}\.", basename)
    if not match:
        raise ValueError(f"Could not parse sensing date from {basename}.")
    
    return datetime.strptime(match.group(1), "%Y%m%d").date()

# read each .nc file and extract science variables, lat/lon, and l2_flags
# PACE L2 files have a grouped NetCDF structure:
# varname in geophysical_data
# latitude in navigation_data
# longitude in navigation_data
# l2_flags in geophysical_data
def read_pace_granule(nc_path, suite):

    result = {}

    with nc.Dataset(nc_path) as root:
        # lat/lon are in the navigation_data group
        nav = root.groups["navigation_data"]
        lat = nav.variables["latitude"][:]
        lon = nav.variables["longitude"][:]
        result["lat"] = lat
        result["lon"] = lon

        # science variables and l2_flags are in the geophysical_data group
        geo = root.groups["geophysical_data"]

        varmap = science_vars.get(
            short_name_BGC if suite == "bgc" else short_name_PAR, {}
        )
        data_arrays = {}
        for file_varname, friendly_name in varmap.items():
            if file_varname in geo.variables:
                raw = geo.variables[file_varname][:]

                # apply scale_factor and add_offset if present
                v = geo.variables[file_varname]
                scale = getattr(v, "scale_factor", 1.0)
                offset = getattr(v, "add_offset", 0.0)
                data = raw.astype(np.float32) * float(scale) + float(offset)

                if file_varname == "ipar_planar_above":
                    data = data * np.float32(1e6)                
                
                data_arrays[friendly_name] = data
            else:
                print(f"'{file_varname}' not found in {os.path.basename(nc_path)}.")

        # quality flags: read bitmask and store flag_masks and flag_meanings for later use
        if "l2_flags" in geo.variables:
            flags_var = geo.variables["l2_flags"]
            result["l2_flags"] = flags_var[:]
            result["flag_masks"] = flags_var.flag_masks
            result["flag_meanings"] = flags_var.flag_meanings.split()
        else:
            print(f"l2_flags not found in {os.path.basename(nc_path)}.")
            result["l2_flags"] = None
            result["flag_masks"] = []
            result["flag_meanings"] = []

    result["data_arrays"] = data_arrays
    return result

# write a raw (unmasked) NetCDF with embedded lat/lon
# PACE has everything in one file so grids always match - no need to cross-check like Sen-3 OLCI files
def write_swath_netcdf(granule_data, granule_name, suite, swath_output_folder):

    os.makedirs(swath_output_folder, exist_ok=True)
    out_path = os.path.join(swath_output_folder, f"{granule_name}_coordsembedded.nc")

    if os.path.exists(out_path):
        print(f"Raw lat/lon embedded NetCDF already exists, skipping: {os.path.basename(out_path)}.")
        return out_path

    lat = granule_data["lat"]
    lon = granule_data["lon"]
    rows, cols = lat.shape

    print(f"Grid shape: {lat.shape}")

    with nc.Dataset(out_path, "w", format="NETCDF4") as ds:
        ds.title = f"PACE OCI L2 swath (raw, unmasked) — {granule_name}"
        ds.source = "NASA PACE OCI Level-2, NASA OB.DAAC"
        ds.Conventions = "CF-1.8"
        ds.crs = "WGS84 geographic (EPSG:4326) — swath coordinates"
        ds.quality_masking = "None — raw unmasked file"
        ds.suite = suite.upper()

        ds.createDimension("rows", rows)
        ds.createDimension("cols", cols)

        lat_var = ds.createVariable("latitude", "f4", ("rows", "cols"), zlib=True)
        lat_var.units = "degrees_north"
        lat_var.long_name = "pixel latitude"
        lat_var.standard_name = "latitude"
        lat_var[:] = lat.data if np.ma.is_masked(lat) else lat

        lon_var = ds.createVariable("longitude", "f4", ("rows", "cols"), zlib=True)
        lon_var.units = "degrees_east"
        lon_var.long_name = "pixel longitude"
        lon_var.standard_name = "longitude"
        lon_var[:] = lon.data if np.ma.is_masked(lon) else lon

        for varname, data in granule_data["data_arrays"].items():
            v = ds.createVariable(varname, "f4", ("rows", "cols"),
                                  zlib=True, fill_value=np.float32(np.nan))
            v.coordinates = "latitude longitude"

            if varname == "CHLOR_A":
                v.long_name = "Chlorophyll-a concentration"
                v.units = "mg m-3"
            elif varname == "PAR":
                v.long_name = "Instantaneous photosynthetically available radiation (above surface)"
                v.units = "microeinstein m-2 s-1" # after x10^6 conversion
                v.comment = (
                    "Converted from Einstein m-2 s-1 (native PACE units) to microeinstein m-2 s-1."
                    "Instantaneous value at overpass time — directly comparable to OLCI PAR."
                )

            v[:] = (data.filled(np.nan) if np.ma.is_masked(data) else data).astype(np.float32)

    print(f"Raw lat/lon embedded NetCDF written: {os.path.basename(out_path)}.")
    return out_path


# Step 2: Apply l2_flags to created quality-masked NetCDFs

# look up the bitmask value for a named flag
def get_flag_bit(flag_masks, flag_meanings, flag_name):

    if flag_name not in flag_meanings:
        raise ValueError(
            f"'{flag_name}' not found in l2_flags."
            f"Available: {flag_meanings}"
        )
    return int(flag_masks[flag_meanings.index(flag_name)])

# build a boolean valid-pixel mask
# valid pixels do not have any common invalid flags and variable-specific flags
# TRUE = include pixel, FALSE = exclude pixel
def build_quality_mask_pace(l2_flags, flag_masks, flag_meanings, varname):
    
    # start with all pixels being valid
    invalid_mask = np.zeros(l2_flags.shape, dtype=bool)

    for flag_name in common_invalid_flags:
        try:
            bit = get_flag_bit(flag_masks, flag_meanings, flag_name)
            invalid_mask |= (l2_flags & bit) != 0
        except ValueError as e:
            print(f"Warning: {e}")

    for flag_name in variable_flags.get(varname, []):
        try:
            bit = get_flag_bit(flag_masks, flag_meanings, flag_name)
            invalid_mask |= (l2_flags & bit) != 0
        except ValueError as e:
            print(f"Warning: {e}")

    return ~invalid_mask  # TRUE = valid

# apply l2_flags and produces quality-masked NetCDFs
# raw files are never modified
def apply_quality_masks_pace(granule_data, raw_swath_path, granule_name, masked_output_folder):
    
    if granule_data["l2_flags"] is None:
        print(f"Skipping masking — no l2_flags available for {granule_name}.")
        return None

    os.makedirs(masked_output_folder, exist_ok=True)
    out_path = os.path.join(masked_output_folder, f"{granule_name}_qualitymasked.nc")

    if os.path.exists(out_path):
        print(f"Quality-masked NetCDF already exists, skipping: {os.path.basename(out_path)}.")
        return out_path

    print(f"Applying l2_flags quality masks.")

    l2_flags = granule_data["l2_flags"]
    flag_masks = granule_data["flag_masks"]
    flag_meanings = granule_data["flag_meanings"]

    with nc.Dataset(raw_swath_path) as raw_ds:
        lat = raw_ds.variables["latitude"][:]
        lon = raw_ds.variables["longitude"][:]
        rows, cols = lat.shape

        with nc.Dataset(out_path, "w", format="NETCDF4") as out_ds:
            out_ds.title = f"PACE OCI L2 swath (quality-masked) — {granule_name}"
            out_ds.source = "NASA PACE OCI Level-2, NASA OB.DAAC"
            out_ds.Conventions = "CF-1.8"
            out_ds.crs = "WGS84 geographic (EPSG:4326) — swath coordinates"
            out_ds.quality_masking = (
                "Source: https://oceancolor.gsfc.nasa.gov/resources/atbd/ocl2flags/"
            )

            out_ds.createDimension("rows", rows)
            out_ds.createDimension("cols", cols)

            lat_var = out_ds.createVariable("latitude", "f4", ("rows", "cols"), zlib=True)
            lat_var.units = "degrees_north"
            lat_var.long_name = "pixel latitude"
            lat_var.standard_name = "latitude"
            lat_var[:] = lat.data if np.ma.is_masked(lat) else lat

            lon_var = out_ds.createVariable("longitude", "f4", ("rows", "cols"), zlib=True)
            lon_var.units = "degrees_east"
            lon_var.long_name = "pixel longitude"
            lon_var.standard_name = "longitude"
            lon_var[:] = lon.data if np.ma.is_masked(lon) else lon

            varnames = [v for v in raw_ds.variables if v not in ("latitude", "longitude")]

            for varname in varnames:
                data = raw_ds.variables[varname][:].astype(np.float32)
                valid = build_quality_mask_pace(l2_flags, flag_masks, flag_meanings, varname)

                masked_data = np.where(valid, data, np.nan).astype(np.float32)

                n_total = valid.size
                n_valid = int(np.sum(valid))
                n_invalid = n_total - n_valid
                pct_valid = n_valid / n_total * 100

                v = out_ds.createVariable(varname, "f4", ("rows", "cols"),
                                          zlib=True, fill_value=np.float32(np.nan))
                v.coordinates = "latitude longitude"
                for attr in raw_ds.variables[varname].ncattrs():
                    if attr != "_FillValue":
                        setattr(v, attr, getattr(raw_ds.variables[varname], attr))
                v[:] = masked_data

                print(f"{varname}: {n_valid}/{n_total} valid pixels "
                      f"({n_invalid} masked, {pct_valid:.1f}% valid)")

                if pct_valid < 10:
                    print(f"Warning: only {pct_valid:.1f}% valid — possible sea ice / polar atmospheric correction issues.")

    print(f"Quality-masked NetCDF written: {os.path.basename(out_path)}")
    return out_path

# Step 3: Extract valid pixels within AOI bounding box from quality-masked NetCDFs

def extract_aoi_pixels(masked_nc_path,
                       lat_min=aoi_lat_min, lat_max=aoi_lat_max,
                       lon_min=aoi_lon_min, lon_max=aoi_lon_max):
    
    with nc.Dataset(masked_nc_path) as ds:
        lat = ds.variables["latitude"][:]
        lon = ds.variables["longitude"][:]

        aoi_mask = (
            (lat >= lat_min) & (lat <= lat_max) &
            (lon >= lon_min) & (lon <= lon_max)
        )
        n_aoi = int(np.sum(aoi_mask))
        if n_aoi == 0:
            print(f"No pixels found within AOI for {os.path.basename(masked_nc_path)}")
            return None

        print(f"AOI pixels found: {n_aoi}")

        results = {}
        varnames = [v for v in ds.variables if v not in ("latitude", "longitude")]
        for varname in varnames:
            data = ds.variables[varname][:].astype(np.float32)
            aoi_values = data[aoi_mask]
            valid_values = aoi_values[~np.isnan(aoi_values)]
            n_valid = len(valid_values)
            if n_valid == 0:
                print(f"{varname}: no valid pixels in AOI after quality masking")
            else:
                print(f"{varname}: {n_valid} valid pixels in AOI")
            results[varname] = valid_values

    return results

# Step 4: Calculate daily and weekly statistics

# iterate over all quality-masked NetCDFs, extract AOI pixels
# then compute daily and weekly statistics
# multiple granules on the same day are pooled, so each day gets a 1D list of values per variable
def compute_statistics(masked_output_folder,
                       lat_min=aoi_lat_min, lat_max=aoi_lat_max,
                       lon_min=aoi_lon_min, lon_max=aoi_lon_max):
    
    masked_files = sorted([
        os.path.join(masked_output_folder, f)
        for f in os.listdir(masked_output_folder)
        if f.endswith("_qualitymasked.nc")
    ])

    if not masked_files:
        print("No quality-masked NetCDF files found.")
        return None, None

    print(f"\nFound {len(masked_files)} quality-masked NetCDF(s) to process.\n")

    daily_values = defaultdict(lambda: defaultdict(list))

    for swath_path in masked_files:
        print(f"Extracting AOI pixels: {os.path.basename(swath_path)}")
        try:
            date = parse_sensing_date_pace(swath_path)
        except ValueError as e:
            print(f"Warning: {e}")
            continue

        pixels = extract_aoi_pixels(swath_path, lat_min, lat_max, lon_min, lon_max)
        if pixels is None:
            continue

        for varname, values in pixels.items():
            if len(values) > 0:
                daily_values[date][varname].extend(values.tolist())

    # daily statistics
    daily_stats = {}
    for date, var_dict in sorted(daily_values.items()):
        daily_stats[date] = {}
        for varname, values in var_dict.items():
            arr = np.array(values, dtype=np.float32)
            daily_stats[date][varname] = {
                "mean": float(np.nanmean(arr)),
                "median": float(np.nanmedian(arr)),
                "p25": float(np.nanpercentile(arr, 25)),
                "p75": float(np.nanpercentile(arr, 75)),
                "n_pixels": len(arr),
            }

    # weekly statistics (mean of daily means)
    weekly_values = defaultdict(lambda: defaultdict(list))
    for date, var_dict in daily_stats.items():
        iso_week = f"{date.isocalendar()[0]}-W{date.isocalendar()[1]:02d}"
        for varname, s in var_dict.items():
            if not np.isnan(s["mean"]):
                weekly_values[iso_week][varname].append(s["mean"])

    weekly_stats = {}
    for iso_week, var_dict in sorted(weekly_values.items()):
        weekly_stats[iso_week] = {}
        for varname, means in var_dict.items():
            weekly_stats[iso_week][varname] = {
                "mean": float(np.nanmean(means)),
                "n_days": len(means),
            }

    # print summary
    print("\nDaily statistics (McMurdo Sound AOI)")
    for date, var_dict in sorted(daily_stats.items()):
        print(f"{date}")
        for varname, s in var_dict.items():
            print(f"{varname}: mean={s['mean']:.4f}, median={s['median']:.4f}, "
                  f"p25={s['p25']:.4f}, p75={s['p75']:.4f}, n={s['n_pixels']}")

    print("\nWeekly statistics (McMurdo Sound AOI)")
    for iso_week, var_dict in sorted(weekly_stats.items()):
        print(f"{iso_week}")
        for varname, s in var_dict.items():
            print(f"{varname}: mean={s['mean']:.4f}, n_days={s['n_days']}")

    return daily_stats, weekly_stats

# save daily statistics to JSON for visualisation
def save_daily_stats_json(daily_stats, output_folder):

    os.makedirs(output_folder, exist_ok=True)
    out_path = os.path.join(output_folder, "PACE_daily_stats.json")
    serialisable = {str(k): v for k, v in daily_stats.items()}
    with open(out_path, "w") as f:
        json.dump(serialisable, f, indent=2)
    print(f"Daily stats saved: {out_path}")
    return out_path

# Step 5 (optional): Resample quality-masked NetCDFs to projected GeoTIFFs, clipped to the AOI
# pixels outside the AOI bounding box are set to NaN before resampling
# target CRS: EPSG:5479 (RSRGD2000 / MSLC2000), 1.2km resolution (native PACE L2 resolution)

def resample_swath_to_geotiff(masked_swath_path, geotiff_output_folder,
                               target_crs=target_crs,
                               resolution_m=target_res_metres,
                               lat_min=aoi_lat_min, lat_max=aoi_lat_max,
                               lon_min=aoi_lon_min, lon_max=aoi_lon_max):
    
    if not os.path.exists(masked_swath_path):
        print(f"Error: Quality-masked NetCDF not found: {masked_swath_path}")
        return
 
    granule_name = os.path.splitext(os.path.basename(masked_swath_path))[0]
    os.makedirs(geotiff_output_folder, exist_ok=True)
 
    with nc.Dataset(masked_swath_path) as ds:
        lat = ds.variables["latitude"][:]
        lon = ds.variables["longitude"][:]
        varnames = [v for v in ds.variables if v not in ("latitude", "longitude")]
        data_arrays = {v: ds.variables[v][:] for v in varnames}
 
    # mask out pixels outside the AOI before resampling
    aoi_mask = (
        (lat >= lat_min) & (lat <= lat_max) &
        (lon >= lon_min) & (lon <= lon_max)
    )
    n_aoi = int(np.sum(aoi_mask))
    if n_aoi == 0:
        print(f"No pixels within AOI for {granule_name}, skipping GeoTIFF export.")
        return
    print(f"AOI pixels: {n_aoi}")
 
    # mask lat/lon arrays to AOI so the projected grid extent is AOI-bounded
    lat_aoi = np.where(aoi_mask, lat, np.nan)
    lon_aoi = np.where(aoi_mask, lon, np.nan)
 
    transformer = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    x_proj, y_proj = transformer.transform(lon_aoi.ravel(), lat_aoi.ravel())
 
    # compute projected extent from AOI pixels only (ignore NaNs from outside AOI)
    x_proj_valid = x_proj[~np.isnan(x_proj)]
    y_proj_valid = y_proj[~np.isnan(y_proj)]
    x_min, x_max = x_proj_valid.min(), x_proj_valid.max()
    y_min, y_max = y_proj_valid.min(), y_proj_valid.max()
 
    grid_cols = int(np.ceil((x_max - x_min) / resolution_m))
    grid_rows = int(np.ceil((y_max - y_min) / resolution_m))
    print(f"Output grid: {grid_rows} rows x {grid_cols} cols at {resolution_m}m")
 
    # use AOI-clipped lat/lon for the swath definition
    swath_def = geometry.SwathDefinition(lons=lon_aoi, lats=lat_aoi)
    crs_obj = CRS.from_epsg(int(target_crs.split(":")[1]))
    area_def = geometry.AreaDefinition(
        area_id = "mslc2000_grid",
        description = f"MSLC2000 {resolution_m}m grid",
        proj_id = target_crs,
        projection = crs_obj.to_dict(),
        width = grid_cols,
        height = grid_rows,
        area_extent = (x_min, y_min, x_max, y_max)
    )
 
    radius = resolution_m * 2
    transform = from_bounds(x_min, y_min, x_max, y_max, grid_cols, grid_rows)
 
    for varname, data in data_arrays.items():
        out_filename = f"{granule_name}_{varname}.tif"
        out_path = os.path.join(geotiff_output_folder, out_filename)
 
        if os.path.exists(out_path):
            print(f"Already exists, skipping: {out_filename}")
            continue
 
        data_filled = (data.filled(np.nan) if np.ma.is_masked(data) else data).astype(np.float32)
 
        # apply AOI mask — pixels outside the AOI are set to NaN before resampling
        data_filled = np.where(aoi_mask, data_filled, np.nan)
 
        resampled = kd_tree.resample_nearest(
            swath_def, data_filled, area_def,
            radius_of_influence=radius,
            fill_value=np.nan
        )
 
        with rasterio.open(
            out_path, "w",
            driver = "GTiff",
            height = grid_rows,
            width = grid_cols,
            count = 1,
            dtype = np.float32,
            crs = target_crs,
            transform = transform,
            nodata    = np.nan,
            compress  = "lzw",
        ) as dst:
            dst.write(resampled.astype(np.float32), 1)
 
        print(f"Exported GeoTIFF: {out_filename}")

# FULL PROCESSING PIPELINE

# Function to run all downloaded PACE OCI granules through the full pipeline:
# Step 1: Read downloaded NetCDF, write NetCDF with embedded lat/lon
# Step 2: Apply l2_flags masks, write quality-masked NetCDF
# Step 3: Extract AOI pixels (called inside compute_statistics in step 4)
# Step 4: Compute daily and weekly statistics
# Step 5 (optional): Resample to GeoTIFF and reproject to target CRS

def process_all_granules(downloads_folder, swath_output_folder, masked_output_folder,
                         stats_output_folder=None, geotiff_output_folder=None):

    all_nc_files = sorted([
        os.path.join(downloads_folder, f)
        for f in os.listdir(downloads_folder)
        if f.endswith(".nc")
    ])
    
    if not all_nc_files:
        print("No downloaded granule files to process.")
        return None, None

    print(f"\nFound {len(all_nc_files)} granule file(s) to process.\n")

    for nc_path in sorted(all_nc_files):
        nc_path = str(nc_path)  # earthaccess may return Path objects
        granule_name = get_pace_granule_name(nc_path)

        # determine suite from filename
        if "BGC" in granule_name.upper():
            suite = "bgc"
        elif "PAR" in granule_name.upper():
            suite = "par"
        else:
            print(f"Unknown suite for {granule_name}, skipping.")
            continue

        print(f"\nGranule: {granule_name}  (suite: {suite.upper()})")

        # Step 1: Read downloaded files and write NetCDF with embedded lat/lon
        try:
            granule_data = read_pace_granule(nc_path, suite)
        except Exception as e:
            print(f"Error reading {granule_name}: {e}")
            continue

        raw_path = write_swath_netcdf(granule_data, granule_name, suite, swath_output_folder)
        if raw_path is None:
            continue

        # Step 2: Apply quality masks
        masked_path = apply_quality_masks_pace(
            granule_data, raw_path, granule_name, masked_output_folder
        )

        # Step 5 (optional): resample to GeoTIFF
        if geotiff_output_folder and masked_path:
            resample_swath_to_geotiff(masked_path, geotiff_output_folder)

    # Steps 3 and 4: Extract AOI pixels and compute statistics from all masked files
    daily_stats, weekly_stats = compute_statistics(masked_output_folder)

    # save daily stats to JSON
    if stats_output_folder and daily_stats:
        save_daily_stats_json(daily_stats, stats_output_folder)

    return daily_stats, weekly_stats

In [24]:
# MAIN CODE #

# setting downloads_folder again in case this cell is run independently of the earlier code
downloads_folder = "/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_0_RAWDATA/PACE"

swath_output_folder  = "/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_PROCESSED/PACE/netcdf_withcoords_raw"
masked_output_folder = "/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_PROCESSED/PACE/netcdf_wqsfmasked"
stats_output_folder  = "/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_PROCESSED/PACE/jsons_dailystats"
 
# set to None to skip GeoTIFF export or provide a path to enable it
# geotiff_output_folder = None
geotiff_output_folder  = "/Users/gwyneth/Desktop/Masters_Thesis/LEVEL_1_PROCESSED/PACE/geotiffs_resampled"
 
daily_stats, weekly_stats = process_all_granules(
    downloads_folder,
    swath_output_folder,
    masked_output_folder,
    stats_output_folder,
    geotiff_output_folder,
)


Found 120 granule file(s) to process.


Granule: PACE_OCI.20251201T004630.L2.OC_BGC.V3_1  (suite: BGC)
Raw lat/lon embedded NetCDF already exists, skipping: PACE_OCI.20251201T004630.L2.OC_BGC.V3_1_coordsembedded.nc.
Quality-masked NetCDF already exists, skipping: PACE_OCI.20251201T004630.L2.OC_BGC.V3_1_qualitymasked.nc.
AOI pixels: 3446
Output grid: 143 rows x 124 cols at 1200m
Exported GeoTIFF: PACE_OCI.20251201T004630.L2.OC_BGC.V3_1_qualitymasked_CHLOR_A.tif

Granule: PACE_OCI.20251201T004630.L2.PAR.V3_1  (suite: PAR)


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Raw lat/lon embedded NetCDF already exists, skipping: PACE_OCI.20251201T004630.L2.PAR.V3_1_coordsembedded.nc.
Quality-masked NetCDF already exists, skipping: PACE_OCI.20251201T004630.L2.PAR.V3_1_qualitymasked.nc.
AOI pixels: 3446
Output grid: 143 rows x 124 cols at 1200m


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Exported GeoTIFF: PACE_OCI.20251201T004630.L2.PAR.V3_1_qualitymasked_PAR.tif

Granule: PACE_OCI.20251201T005130.L2.OC_BGC.V3_1  (suite: BGC)
Raw lat/lon embedded NetCDF already exists, skipping: PACE_OCI.20251201T005130.L2.OC_BGC.V3_1_coordsembedded.nc.
Quality-masked NetCDF already exists, skipping: PACE_OCI.20251201T005130.L2.OC_BGC.V3_1_qualitymasked.nc.
AOI pixels: 10
Output grid: 4 rows x 20 cols at 1200m


...

Raw lat/lon embedded NetCDF already exists, skipping: PACE_OCI.20251205T142935.L2.OC_BGC.V3_1_coordsembedded.nc.
Quality-masked NetCDF already exists, skipping: PACE_OCI.20251205T142935.L2.OC_BGC.V3_1_qualitymasked.nc.
AOI pixels: 4439
Output grid: 143 rows x 124 cols at 1200m


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Exported GeoTIFF: PACE_OCI.20251205T142935.L2.OC_BGC.V3_1_qualitymasked_CHLOR_A.tif

Granule: PACE_OCI.20251205T142935.L2.PAR.V3_1  (suite: PAR)
Raw lat/lon embedded NetCDF already exists, skipping: PACE_OCI.20251205T142935.L2.PAR.V3_1_coordsembedded.nc.
Quality-masked NetCDF already exists, skipping: PACE_OCI.20251205T142935.L2.PAR.V3_1_qualitymasked.nc.
AOI pixels: 4439
Output grid: 143 rows x 124 cols at 1200m
Exported GeoTIFF: PACE_OCI.20251205T142935.L2.PAR.V3_1_qualitymasked_PAR.tif

Found 120 quality-masked NetCDF(s) to process.

Extracting AOI pixels: PACE_OCI.20251201T004630.L2.OC_BGC.V3_1_qualitymasked.nc
AOI pixels found: 3446


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


CHLOR_A: no valid pixels in AOI after quality masking
Extracting AOI pixels: PACE_OCI.20251201T004630.L2.PAR.V3_1_qualitymasked.nc
AOI pixels found: 3446
PAR: no valid pixels in AOI after quality masking
Extracting AOI pixels: PACE_OCI.20251201T005130.L2.OC_BGC.V3_1_qualitymasked.nc
AOI pixels found: 10
CHLOR_A: no valid pixels in AOI after quality masking
Extracting AOI pixels: PACE_OCI.20251201T005130.L2.PAR.V3_1_qualitymasked.nc
AOI pixels found: 10
PAR: no valid pixels in AOI after quality masking
Extracting AOI pixels: PACE_OCI.20251201T022450.L2.OC_BGC.V3_1_qualitymasked.nc
AOI pixels found: 8454
CHLOR_A: 37 valid pixels in AOI
Extracting AOI pixels: PACE_OCI.20251201T022450.L2.PAR.V3_1_qualitymasked.nc
AOI pixels found: 8454
PAR: no valid pixels in AOI after quality masking
Extracting AOI pixels: PACE_OCI.20251201T022950.L2.OC_BGC.V3_1_qualitymasked.nc
No pixels found within AOI for PACE_OCI.20251201T022950.L2.OC_BGC.V3_1_qualitymasked.nc
Extracting AOI pixels: PACE_OCI.20251201